In [1]:
# --- Step 1: Load Libraries ---
import os
import json
import pandas as pd
from dotenv import load_dotenv

# --- Step 2: Load .env ---
load_dotenv()

# --- Step 3: Load config.json ---
with open("config/config.json", "r") as f:
    config = json.load(f)

# --- Step 4: Load transcripts ---
df = pd.read_csv("data/transcripts.csv")

print("✅ Data Loaded")
print(df.head(), "\n")

# --- Step 5: Initialize LLM ---
from langchain_openai import ChatOpenAI
from langchain_google_genai import ChatGoogleGenerativeAI

def load_llm(config):
    provider = "openai"
    
    print(provider)

    if provider == "openai":
        llm = ChatOpenAI(
            model=config["llm"]["openai_model"],
            temperature=config["llm"]["temperature"]
        )

    elif provider == "gemini":
        llm = ChatGoogleGenerativeAI(
            model=config["llm"]["gemini_model"],
            temperature=config["llm"]["temperature"]
        )

    else:
        raise ValueError("Invalid LLM_PROVIDER")

    return llm

llm = load_llm(config)

print(f"✅ LLM Initialized using provider: {os.getenv('LLM_PROVIDER')}")

# --- Step 6: Test LLM ---
response = llm.invoke("Say 'setup successful' in one short sentence.")
print("\n🧠 LLM Response:")
print(response.content)

✅ Data Loaded
   call_id agent_name                                         transcript  \
0        1      Alice  Customer: I was charged twice for my premium t...   
1        2        Bob  Customer: My claim has been pending for 2 week...   
2        3    Charlie  Customer: This is the third time I'm calling! ...   
3        4      Diana  Customer: Can you explain my coverage details?...   

  expected_call_type  
0            billing  
1             claims  
2          complaint  
3      general_query   

openai
✅ LLM Initialized using provider: None

🧠 LLM Response:
Setup successful.


In [2]:
# --- Step 1: Imports ---
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import PydanticOutputParser
from pydantic import BaseModel, Field

# --- Step 2: Define Output Schema ---
class ClassificationOutput(BaseModel):
    call_type: str = Field(description="Type of customer call")
    confidence: float = Field(description="Confidence score between 0 and 1")

parser = PydanticOutputParser(pydantic_object=ClassificationOutput)

# --- Step 3: Prompt Template ---
prompt = PromptTemplate(
    template="""
You are a call classification assistant.

Classify the following customer support transcript into one of these categories:
{labels}

Transcript:
{transcript}

{format_instructions}
""",
    input_variables=["transcript"],
    partial_variables={
        "format_instructions": parser.get_format_instructions(),
        "labels": config["classification"]["labels"]
    }
)

# --- Step 4: Create Chain ---
classification_chain = prompt | llm | parser

# --- Step 5: Test with one transcript ---
sample_text = df.iloc[0]["transcript"]

result = classification_chain.invoke({
    "transcript": sample_text
})

print("✅ Classification Result:")
print(result)

✅ Classification Result:
call_type='billing' confidence=0.95


In [3]:
from tqdm import tqdm

results = []

for i, row in tqdm(df.iterrows(), total=len(df), desc="Classifying Calls"):
    try:
        output = classification_chain.invoke({
            "transcript": row["transcript"]
        })
        
        results.append({
            "call_id": row["call_id"],
            "predicted_call_type": output.call_type,
            "confidence": output.confidence
        })
        
    except Exception as e:
        print(f"❌ Error at row {i}: {e}")
        results.append({
            "call_id": row["call_id"],
            "predicted_call_type": None,
            "confidence": None
        })

# Convert to DataFrame
results_df = pd.DataFrame(results)

# Merge with original
df = df.merge(results_df, on="call_id")

print("\n✅ Batch Classification Completed\n")
print(df[["call_id", "expected_call_type", "predicted_call_type", "confidence"]])

Classifying Calls: 100%|██████████| 4/4 [00:07<00:00,  1.77s/it]


✅ Batch Classification Completed

   call_id expected_call_type predicted_call_type  confidence
0        1            billing             billing        0.95
1        2             claims              claims        0.95
2        3          complaint           complaint        0.85
3        4      general_query       general_query        0.90


In [4]:
# --- Step 1: Define routing logic ---

def route_call(call_type):
    if call_type == "billing":
        return ["knowledge_accuracy", "resolution_quality"]
    
    elif call_type == "claims":
        return ["knowledge_accuracy", "resolution_quality"]
    
    elif call_type == "complaint":
        return ["tone_empathy", "resolution_quality"]
    
    elif call_type == "general_query":
        return ["knowledge_accuracy"]
    
    else:
        return ["knowledge_accuracy"]  # fallback


# --- Step 2: Apply routing to dataset ---

df["evaluation_plan"] = df["predicted_call_type"].apply(route_call)

print("✅ Routing Applied\n")
print(df[["call_id", "predicted_call_type", "evaluation_plan"]])

✅ Routing Applied

   call_id predicted_call_type                           evaluation_plan
0        1             billing  [knowledge_accuracy, resolution_quality]
1        2              claims  [knowledge_accuracy, resolution_quality]
2        3           complaint        [tone_empathy, resolution_quality]
3        4       general_query                      [knowledge_accuracy]


In [5]:
# --- Step 1: Imports ---
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import PydanticOutputParser
from pydantic import BaseModel, Field

# --- Step 2: Define Output Schema ---
class ToneEvaluation(BaseModel):
    score: int = Field(description="Score between 1 and 5")
    reasoning: str = Field(description="Explanation of the score")

tone_parser = PydanticOutputParser(pydantic_object=ToneEvaluation)

# --- Step 3: Prompt Template ---
tone_prompt = PromptTemplate(
    template="""
You are a QA evaluator for customer support calls.

Evaluate the agent's tone and empathy in the following transcript.

Consider:
- Did the agent acknowledge the customer's issue?
- Was the tone polite and professional?
- Did the agent show empathy?

Transcript:
{transcript}

{format_instructions}
""",
    input_variables=["transcript"],
    partial_variables={
        "format_instructions": tone_parser.get_format_instructions()
    }
)

# --- Step 4: Chain ---
tone_chain = tone_prompt | llm | tone_parser

# --- Step 5: Test on ONE complaint call (important)
sample_text = df[df["predicted_call_type"] == "complaint"].iloc[0]["transcript"]

result = tone_chain.invoke({
    "transcript": sample_text
})

print("✅ Tone Evaluation Result:")
print(result)

✅ Tone Evaluation Result:
score=3 reasoning="The agent acknowledged the customer's issue by apologizing for the inconvenience, which indicates some level of recognition of the customer's frustration. However, the response was quite brief and lacked depth in empathy. While the tone was polite, it could have been more supportive and understanding given that the customer has called multiple times without resolution."


In [6]:
# --- Step 1: Schema ---
class ResolutionEvaluation(BaseModel):
    score: int = Field(description="Score between 1 and 5")
    reasoning: str = Field(description="Explanation of the score")

resolution_parser = PydanticOutputParser(pydantic_object=ResolutionEvaluation)

# --- Step 2: Prompt ---
resolution_prompt = PromptTemplate(
    template="""
You are a QA evaluator for customer support calls.

Evaluate the resolution quality of the agent.

Consider:
- Did the agent fully resolve the customer's issue?
- Were next steps clearly communicated?
- Did the agent confirm resolution before ending?

Transcript:
{transcript}

{format_instructions}
""",
    input_variables=["transcript"],
    partial_variables={
        "format_instructions": resolution_parser.get_format_instructions()
    }
)

# --- Step 3: Chain ---
resolution_chain = resolution_prompt | llm | resolution_parser

# --- Step 4: Test
sample_text = df.iloc[0]["transcript"]

result = resolution_chain.invoke({
    "transcript": sample_text
})

print("✅ Resolution Evaluation Result:")
print(result)

✅ Resolution Evaluation Result:
score=2 reasoning="The agent did not fully resolve the customer's issue as the transcript ends abruptly without confirming the resolution or providing next steps. There is no indication that the agent addressed the double charge or offered a solution."


In [7]:
# --- Step 1: Imports (safe to repeat) ---
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import PydanticOutputParser
from pydantic import BaseModel, Field

# --- Step 2: Schema ---
class KnowledgeEvaluation(BaseModel):
    score: int = Field(description="Score between 1 and 5")
    reasoning: str = Field(description="Explanation of the score")

knowledge_parser = PydanticOutputParser(pydantic_object=KnowledgeEvaluation)

# --- Step 3: Prompt ---
knowledge_prompt = PromptTemplate(
    template="""
You are a QA evaluator for customer support calls.

Evaluate the agent's knowledge accuracy and clarity.

Consider:
- Did the agent provide correct and relevant information?
- Was the explanation clear and easy to understand?
- Did the agent avoid vague or misleading statements?

IMPORTANT:
- If the transcript does not contain enough information, give a moderate score (2 or 3) and explain why.

Transcript:
{transcript}

{format_instructions}
""",
    input_variables=["transcript"],
    partial_variables={
        "format_instructions": knowledge_parser.get_format_instructions()
    }
)

# --- Step 4: Chain ---
knowledge_chain = knowledge_prompt | llm | knowledge_parser

print("✅ Knowledge Chain Ready")

✅ Knowledge Chain Ready


In [8]:
from tqdm import tqdm

# --- Step 1: Evaluation Runner ---

def run_evaluations(transcript, eval_plan):
    results = {}

    if "tone_empathy" in eval_plan:
        try:
            tone_result = tone_chain.invoke({"transcript": transcript})
            results["tone"] = tone_result.model_dump()
        except Exception as e:
            results["tone"] = {"error": str(e)}

    if "knowledge_accuracy" in eval_plan:
        try:
            knowledge_result = knowledge_chain.invoke({"transcript": transcript})
            results["knowledge"] = knowledge_result.model_dump()
        except Exception as e:
            results["knowledge"] = {"error": str(e)}

    if "resolution_quality" in eval_plan:
        try:
            resolution_result = resolution_chain.invoke({"transcript": transcript})
            results["resolution"] = resolution_result.model_dump()
        except Exception as e:
            results["resolution"] = {"error": str(e)}

    return results


# --- Step 2: Apply to entire dataset ---

evaluation_outputs = []

for i, row in tqdm(df.iterrows(), total=len(df), desc="Running Evaluations"):
    output = run_evaluations(row["transcript"], row["evaluation_plan"])
    
    evaluation_outputs.append({
        "call_id": row["call_id"],
        "evaluation_output": output
    })

# Convert to DataFrame
eval_df = pd.DataFrame(evaluation_outputs)

# Merge
df = df.merge(eval_df, on="call_id")

print("\n✅ Evaluation Completed\n")

# Show one example clearly
import pprint
pprint.pprint(df.iloc[0]["evaluation_output"])

Running Evaluations: 100%|██████████| 4/4 [00:10<00:00,  2.67s/it]


✅ Evaluation Completed

{'knowledge': {'reasoning': 'The transcript provides limited information about '
                            "the agent's response to the customer's issue. "
                            "While the agent acknowledges the customer's "
                            'concern and indicates they will check the issue, '
                            'there is no further detail on the information '
                            'provided or the resolution process. This lack of '
                            'clarity and completeness leads to a moderate '
                            'score.',
               'score': 3},
 'resolution': {'reasoning': "The agent did not fully resolve the customer's "
                             'issue as the transcript ends abruptly without '
                             'confirming the resolution or providing next '
                             'steps. There is no indication that the agent '
                             'addressed the double ch

In [9]:
# --- Step 1: Schema ---
class FinalReport(BaseModel):
    summary: str = Field(description="Overall evaluation summary")
    recommendations: list[str] = Field(description="List of actionable improvements")

final_parser = PydanticOutputParser(pydantic_object=FinalReport)

# --- Step 2: Prompt ---
final_prompt = PromptTemplate(
    template="""
You are a QA manager reviewing customer support calls.

Based on the evaluation results below, generate:

1. A concise summary of the agent's performance
2. A list of actionable recommendations for improvement

Evaluation Data:
{evaluation_output}

IMPORTANT:
- Be specific and practical
- Do not repeat scores
- Focus on improvement

{format_instructions}
""",
    input_variables=["evaluation_output"],
    partial_variables={
        "format_instructions": final_parser.get_format_instructions()
    }
)

# --- Step 3: Chain ---
final_chain = final_prompt | llm | final_parser

# --- Step 4: Test on one row
sample_eval = df.iloc[0]["evaluation_output"]

result = final_chain.invoke({
    "evaluation_output": sample_eval
})

print("✅ Final QA Report:")
print(result)

✅ Final QA Report:
summary="The agent demonstrated a basic acknowledgment of the customer's concern but failed to provide sufficient information or clarity regarding the issue. The interaction ended without confirming a resolution or outlining next steps, indicating a need for improvement in both knowledge and resolution skills." recommendations=['Enhance product knowledge to provide detailed and accurate information regarding common issues, such as billing discrepancies.', 'Implement a structured approach to ensure all customer concerns are addressed, including confirming resolution and next steps before ending the call.', 'Encourage active listening techniques to better understand customer issues and respond appropriately.', 'Provide training on effective communication skills to improve clarity and completeness in responses.', 'Utilize call scripts or guidelines that prompt agents to confirm resolution and summarize the conversation before concluding the call.']


In [10]:
from tqdm import tqdm

final_outputs = []

for i, row in tqdm(df.iterrows(), total=len(df), desc="Generating Final Reports"):
    try:
        result = final_chain.invoke({
            "evaluation_output": row["evaluation_output"]
        })

        final_outputs.append({
            "call_id": row["call_id"],
            "summary": result.summary,
            "recommendations": result.recommendations
        })

    except Exception as e:
        print(f"❌ Error at row {i}: {e}")
        final_outputs.append({
            "call_id": row["call_id"],
            "summary": None,
            "recommendations": None
        })

# Convert to DataFrame
final_df = pd.DataFrame(final_outputs)

# Merge
df = df.merge(final_df, on="call_id")

print("\n✅ Final Reports Generated\n")

# Show final result
df[[
    "call_id",
    "predicted_call_type",
    "evaluation_output",
    "summary",
    "recommendations"
]]

Generating Final Reports: 100%|██████████| 4/4 [00:07<00:00,  1.76s/it]


✅ Final Reports Generated



,call_id,predicted_call_type,evaluation_output,summary,recommendations
0,1,billing,"{'knowledge': {'score': 3, 'reasoning': 'The t...",The agent demonstrated a basic understanding o...,[Enhance knowledge of common issues and resolu...
1,2,claims,"{'knowledge': {'score': 2, 'reasoning': 'The t...",The agent demonstrated insufficient knowledge ...,[Enhance training on claim processes to improv...
2,3,complaint,"{'tone': {'score': 3, 'reasoning': 'The agent ...",The agent demonstrated a polite and profession...,[Enhance empathy by actively listening to the ...
3,4,general_query,"{'knowledge': {'score': 3, 'reasoning': 'The t...",The agent demonstrated a basic understanding o...,[Enhance knowledge of coverage details to prov...


In [11]:
accuracy = (df["expected_call_type"] == df["predicted_call_type"]).mean()
print(f"🎯 Classification Accuracy: {accuracy:.2f}")

🎯 Classification Accuracy: 1.00


In [12]:
df.to_excel("data/output.xlsx", index=False)
print("✅ Results saved to data/output.xlsx")

✅ Results saved to data/output.xlsx
